# Week 4: Temporal-Difference Learning

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab explores Temporal-Difference (TD) learning, which overcomes limitations present in the methods we've learned so far. Like Dynamic Programming (DP), TD is based on bootstrapping; that is, it uses estimates rather than outcomes to learn optimal policy. However, TD does not require a model of environment dynamics to derive estimates; it learns from experience what rewards are obtained from state-action pairs. That experiential approach is shared with Monte Carlo (MC) methods, but where MC learns from complete episodes and final returns, TD dynamically updates estimates after each step using the observed reward and the estimated value of the next state. This means TD can work where other methods are impractical: it doesn't need a model like DP, and it doesn't require complete episodes like MC.

As in prior methods, exploration versus exploitation is still the central trade-off. Greedy optimization risks insufficient exploration. DP can rely on a model to steer learning; when learning is based on experience, an exploration coefficient has to be used instead.

In Chapter 6 of "Reinforcement Learning: An introduction", Sutton & Barto describe two algorithms for addressing this problem (p. 100): SARSA, an "on-policy" approach which "attempt[s] to evaluate or improve the policy that is used to make decisions," and Q-learning, an "off-policy" method which "evaluate\[s] or improve\[s] a policy different from that used to generate the data."

The key difference between SARSA and Q-learning is in the way that they learn (i.e. update Q-values). SARSA learns based on the next action it takes. Q-learning learns based on the max value of the next possible actions. Where SARSA asks, "what happens if I sometimes act randomly", Q-learning asks, "what happens if I always act optimally."

This lab uses the Cliff Walking Gymnasium environment to understand the differences between SARSA and Q-learning, and the effects that they produce. The Cliff Walking environment is a 4x12 grid, with 37 reachable observation states, 10 "over the cliff" states, and one goal state.

From each cell, the agent can take one of four actions: up, down, right, and left. The agent incurs a penalty of -1 for each step it takes in the 37 reachable "solid ground" cells. Stepping into one of the 10 "over the cliff" states incurs a penalty of -100 and resets the agent back to start. Reaching the goal state terminates the episode.

We will implement a SARSA agent and a Q-learning agent. Both agents will sometimes fall off the cliff and incur the -100 penalty. The main difference is that SARSA will merge those penalties into the Q-values; Q-learning's max operator will preserve the more optimistic estimates. By this mechanism, we can expect the SARSA agent to learn to avoid the row next to the cliff, but we can expect the Q-learning agent to learn that the row next to the cliff is the shortest path to the goal, even if it is the one with the greatest risk.

## Section 2: Deliverables

### GitHub Repository

GitHub Repository found here:  https://github.com/craig-rudman/MSDS684/tree/main/W4

### Implementation Summary

This project implements SARSA (on-policy) and Q-learning (off-policy) temporal-difference agents. Experiments play out in the CliffWalking-v1 Gymnasium 4x12 grid environment. A baseline is established with α=0.1, ε=0.1, no decay. The baseline, and all subsequent experiments are configured to run 500 episodes with 30 random seeds. Sweeps across α values [0.05, 0.3, 0.5] and ε schedules for linear and exponential decay are tested and compared against the baseline and one another. SARSA outperforms Q-learning in training, but Q-learning follows the shorter optimal path at test time.

Note: the assignment specifies CliffWalking-v0, but as that is deprecated, I had to go with v1.

#### Key Results and Analysis

##### Policy Comparison

The results of this project demonstrate both the mechanism and effects of the temporal-difference (TD) algorithms SARSA and Q-learning described in Sutton & Barto, Chapter 6.

The SARSA action-value function for on-policy TD control updates estimates using actual next state-action values $(s',a')$ to find the TD error, adjusted by α (the learning rate) and discount factor γ (Sutton and Barto, p.130). 

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma Q(s', a') - Q(s,a) \right]$$

<figure>
    <img src="../results/policy_arrows_SARSA α=0.1 ε=0.1.png" width="80%">
    <figcaption>Figure 1: The SARSA agent learns to take a safer path two rows above the cliff.</figcaption>
</figure>

By contrast, the Q-learning off-policy action-value function for TD control updates estimates using the maximum value of all possible next state-action values $\max_a Q(s', a)$ to find the TD error (Sutton and Barto, p.131).

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_a Q(s', a) - Q(s,a) \right]$$

<figure>
    <img src="../results/policy_arrows_Q-Learning α=0.1 ε=0.1.png" width="80%">
    <figcaption>Figure 2: The Q-learning agent learns to take the shorter path adjacent to the cliff wall.</figcaption>
</figure>

Note that CliffWalking is episodic; rewards are undiscounted (i.e. γ=1 for both algorithms).

##### Learning Process

While training, SARSA outperforms Q-learning. That is, the SARSA agent falls over the cliff edge less than the Q-learning agent, so the mean episode return of the SARSA agent is less negative than that of the Q-learning agent. Figure 3 below shows a comparison of mean episode return for the SARSA and Q-learning agent over 500 episodes.  The final 10% of returns are averaged to evaluate performance. The SARSA agent averaged -22.24 (95% CI ±1.23) compared to the Q-learning agent, which averaged -48.92 (±3.89).

<figure>
    <img src="../results/baseline_learning_curves.png" width="80%">
    <figcaption>Figure 3: Learning curves of SARSA vs. Q-learning agents, with α=0.1 and ε=0.1, no decay.</figcaption>
</figure>

This result is expected. Sutton and Barto wrote, "Although Q-learning actually learns the values of the optimal policy, its online performance is worse than that of Sarsa, which learns the roundabout policy" (Sutton and Barto, p. 132).  We can see in Figure 4 that, despite Q-learning's performance as measured in cumulative reward, it did, in fact, learn the optimal policy (i.e. the shortest path to the goal).

<figure>
    <img src="../results/trajectory_comparison.png" width="80%">
    <figcaption>Figure 4: Learned trajectories of SARSA vs. Q-learning agents. SARSA takes 15 steps via the safer inland route; Q-learning takes 13 steps via the cliff edge.</figcaption>
</figure>

Looking at the Q-tables shows us what the agents are learning.  In Figure 5, each cell in the Q-table for each agent reflects the best cumulative return expected, were the agent positioned at that cell. The color gradient is intended to show which cells are more or less valuable. I had expected this to be more illustrative than it turned out to be, but the range of values (-15 compared to -13) is just too small for the color gradient to clearly show the differences.  Adding the values as text helps, but the heatmaps are not as useful as the trajectory maps in showing the difference.

<figure>
    <img src="../results/value_heatmap_SARSA α=0.1 ε=0.1.png" width="80%">
    <img src="../results/value_heatmap_Q-Learning α=0.1 ε=0.1.png" width="80%">
    <figcaption>Figure 5: Q-table values of SARSA and Q-learning agents after 500 episodes, α=0.1 and ε=0.1, no decay.</figcaption>
</figure>

##### Hyperparameter Effects

Two experimental sweeps were conducted to illustrate the effects of the alpha `α` and epsilon `ε` parameters on training. 

**Learning Rate (α)**

Alpha `α` is the step-size parameter which influences the rate of learning. A lower `α` value reflects a slower, more stable learning rate; higher `α` values produce faster learning with higher variance. The value of `α` modulates the amount by which estimates are adjusted with each step; smaller steps are slower but more stable. Sutton and Barto assert that a decay in the learning rate is required to guarantee model convergence (Sutton and Barto, p. 33).  However, we hold the learning rate at a constant (i.e. no decay schedule) to more clearly demonstrate the effect of changing `α` value.  

Figure 6 shows a comparison of learning curves with different `α` values for each agent.

<figure>
    <img src="../results/SARSA_alpha_sweep.png" width="80%">
    <img src="../results/Q-Learning_alpha_sweep.png" width="80%">
    <figcaption>Figure 6: Learning curves for varied α values (ε=0.1 constant). Higher α values converge faster but with greater variance; α=0.05 lags throughout.</figcaption>
</figure>

Table 1 shows the results of these sweeps. I expected that smaller `α` values would produce tighter confidence intervals; that is, more stable learning across seeds. The data does not support this.


In [3]:
import pandas as pd
from IPython.display import Markdown

df = pd.read_csv('../results/alpha_variance.csv').sort_values(by=["Agent","α"], ascending=[False, True])

fmt = {'final_mean': '{:.2f}', '95%_CI': '{:.2f}'}
for col, f in fmt.items():
    if col in df.columns:
        df[col] = df[col].apply(lambda x: f.format(x))

header = '| ' + ' | '.join(df.columns) + ' |'
sep    = '| ' + ' | '.join(['---'] * len(df.columns)) + ' |'
rows   = ['| ' + ' | '.join(str(v) for v in row) + ' |' for _, row in df.iterrows()]
display(Markdown('\n'.join([header, sep] + rows)))

print("Table 1: Final 10% mean episode return and 95% CI by agent and α value (ε=0.1 constant, 30 seeds, 500 episodes), sorted by 95% CI ascending.")


| Agent | α | final_mean | 95%_CI |
| --- | --- | --- | --- |
| SARSA | 0.05 | -27.04 | 1.36 |
| SARSA | 0.1 | -22.24 | 1.23 |
| SARSA | 0.3 | -23.02 | 1.36 |
| SARSA | 0.5 | -26.13 | 2.16 |
| Q-Learning | 0.05 | -49.43 | 3.10 |
| Q-Learning | 0.1 | -48.92 | 3.89 |
| Q-Learning | 0.3 | -51.73 | 3.10 |
| Q-Learning | 0.5 | -49.10 | 3.23 |

Table 1: Final 10% mean episode return and 95% CI by agent and α value (ε=0.1 constant, 30 seeds, 500 episodes), sorted by 95% CI ascending.


For SARSA, α=0.1 is the most stable configuration (CI=±1.23), while the smallest α=0.05 is actually less stable than α=0.1 or α=0.3. For Q-learning, α=0.05 and α=0.3 produce the tightest CI, but the relationship is similarly non-monotonic. It seems that there is no linear correlation between α and confidence interval. 

**Exploration vs Exploitation**

Epsilon `ε` controls the ratio of exploration to exploitation in an ε-soft policy. The baseline training suite held `ε` at a constant ε=0.1, exploiting 90% of the time and exploring 10% of the time. 

Two experiments were conducted to demonstrate the effects of epsilon decay schedules on learning speed, sample efficiency, and final performance. All experimental agents started in the same condition (i.e. α=0.1, ε=0.1) with ε values decaying to a floor of 0.001 over 500 episodes. Figure 7 shows a the effects of linear and exponential decay schedules, with the ε-constant baseline overlayed for comparison.

<figure>
    <img src="../results/SARSA_schedule_sweep.png" width="80%">
    <img src="../results/Q-Learning_schedule_sweep.png" width="80%">
    <figcaption>Figure 7: Comparison of ExponentialDecay and LinearDecay against a baselined constant ε=0.1. </figcaption>
</figure>

In [2]:
import pandas as pd
from IPython.display import Markdown

df = pd.read_csv('../results/combined_summary.csv')

fmt = {
    'final_performance': '{:.2f}',
    'learning_speed':    '{:.0f}',
    'sample_efficiency': '{:.2f}',
    'performance_index': '{:.2f}',
}
for col, f in fmt.items():
    if col in df.columns:
        df[col] = df[col].apply(lambda x: f.format(x))

header = '| ' + ' | '.join(df.columns) + ' |'
sep    = '| ' + ' | '.join(['---'] * len(df.columns)) + ' |'
rows   = ['| ' + ' | '.join(str(v) for v in row) + ' |' for _, row in df.iterrows()]
display(Markdown('\n'.join([header, sep] + rows)))

print('''Table 2: Globally-normalized performance index across all experimental configurations, ranked descending.
Performance index is a weighted composite of final performance (50%), sample efficiency (40%), and learning speed (10%),
where learning speed is the first episode crossing a reward threshold of -75. A higher performance_index is better.''')

| label | final_performance | learning_speed | sample_efficiency | performance_index |
| --- | --- | --- | --- | --- |
| Q-Learning ε=ExponentialDecay | -13.50 | 45 | -43.25 | 0.92 |
| SARSA ε=ExponentialDecay | -15.10 | 52 | -42.88 | 0.90 |
| SARSA ε=LinearDecay | -15.17 | 52 | -46.86 | 0.87 |
| SARSA α=0.3 | -23.02 | 19 | -36.56 | 0.86 |
| SARSA α=0.5 | -26.13 | 12 | -35.66 | 0.83 |
| Q-Learning ε=LinearDecay | -14.85 | 68 | -52.43 | 0.82 |
| SARSA α=0.1 ε=0.1 | -22.24 | 65 | -51.54 | 0.73 |
| SARSA α=0.05 | -27.04 | 116 | -75.92 | 0.45 |
| Q-Learning α=0.5 | -49.10 | 11 | -55.95 | 0.38 |
| Q-Learning α=0.3 | -51.73 | 23 | -58.75 | 0.32 |
| Q-Learning α=0.1 ε=0.1 | -48.92 | 80 | -71.23 | 0.22 |
| Q-Learning α=0.05 | -49.43 | 159 | -89.08 | 0.03 |

Table 2: Globally-normalized performance index across all experimental configurations, ranked descending.
Performance index is a weighted composite of final performance (50%), sample efficiency (40%), and learning speed (10%),
where learning speed is the first episode crossing a reward threshold of -75. A higher performance_index is better.


Decay schedules dominate the top rankings for both agents; reducing late-training exploration amplifies final performance. Q-learning benefits more dramatically than SARSA, jumping from near-bottom to first with ExponentialDecay.  Final performance is near optimal values in training, which is a direct consequence of Q-learning's off-policy updates. The max operator targets the optimal action at every step, so Q-values converge to true optimal estimates regardless of how the behavior policy explores.

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I asked Claude to help me plan a TDD implementation of SARSA and Q-learning on Gymnasium's CliffWalking-v0. My initial prompt was the full lab specification. Rather than generating code, Claude helped me design a modular architecture across seven source files, reviewed it against all lab requirements, and organized a 12-phase roadmap in my development notebook to guide implementation one TODO at a time. No code was written in this session.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration 1

Iteration 1 established the full project scaffold: 8 stub source modules, 6 test files, and 52 tests all failing as expected. During the environment sanity check, the cell raised a hard DeprecatedEnv error for CliffWalking-v0. I asked Claude if we could develop for both v0 and v1. Claude explained that newer Gymnasium versions block v0 entirely and recommended switching to v1. Updated EnvironmentManager and conftest.py accordingly.

Here's how Claude describes what happened:

`Craig came in with a clear philosophy — encapsulation and TDD — and I filled in the details. He named the classes, I proposed the interfaces. He caught the gaps, I fixed them. He added the CLI requirement, I worked out what that meant for the design. The most interesting moment was the reset() method — a small prompt that solved a non-obvious problem in EpisodeRunner before we'd written a line of implementation.`

#### 3.3.2 Iteration 2

Implemented a full vertical slice: TDAgent base class with Q-table initialization, epsilon-greedy action selection, and shared TD update logic. SARSAAgent and QLearningAgent each override _bootstrap — a refactor we made after inspection revealed two identical lines in both subclasses. EnvironmentManager wraps CliffWalking-v1, and EpisodeRunner runs the episode loop across multiple seeds. Validated with 28 passing tests and a behavioral demo showing learning improvement from -515 to -88 mean episode return.

Here's how Claude describes what happened:

`Iteration 2 was where it got interesting. ConstantSchedule went in cleanly, then I violated the TDD workflow by implementing the agents, environment, and runner before confirming the tests failed. Craig caught it immediately. Then during code review, he pushed deeper — asking about alpha, gamma, bootstrapping, the Q-table — and that conversation led to a genuine design improvement. He spotted the duplicated lines in update and asked if we could refactor. The Template Method pattern emerged from that, making the on-policy vs off-policy distinction cleaner than my original implementation. Craig's scrutiny produced better code than I wrote on my own.`


#### 3.3.3 Iteration 3

Implemented LinearDecaySchedule and ExponentialDecaySchedule. Both use an episode counter with epsilon derived on demand via a @property — no stored epsilon value. Linear interpolates between epsilon_start and epsilon_end, clamping at the floor. Exponential applies decay_rate ** episode.

Claude's description:

`Iteration 3 was smoother — the TDD workflow held, red confirmed before a line was written. The most interesting moment came during code review when Craig noticed that value was accessed without parentheses and asked why. That led to a discussion of @property and the design choice to derive epsilon from a counter rather than store it directly. Craig called it elegant, and it is — reset() becomes trivial, and there's no risk of stale state. The one stumble was a stale kernel during the demos, which Craig caught and resolved himself.`

#### 3.3.4 Subsequent Iterations

We iterated a total of 10 cycles, and the pattern was largely the same. During the ninth cycle, I was reviewing the trajectory diagrams and I noticed that the arrows seemed to be indicating a suboptimal, noisy path.  I asked Claude to investigate and we learned the the arrow labels weren't properly mapped.  We fixed the mapping, and the results, accurately reported, aligned to expectations.

### 3.3 Critical Evaluation

As in prior labs, I found that asking Claude to discuss the problem and the solution, rather than implement it was the better approach. That angle yields discussion about key concepts and design decisions, and makes Claude a collaborator rather than an assistant.

When it comes to generating code, a test-first approach works well. It means that each iteration is driven by a specific set of to-do items (literally coded as TODO tags). Making sure the interfaces are well-designed for unit tests helps ensure that the integrations between components never break. Encapsulating code in classes and methods makes sure the solution remains legible, and that there are clear separations of concerns.

Finally, I've started doing code reviews with Claude, asking Claude to walk me through the implementation. Visual inspection catches mistakes.  And every iteration results in a demonstration in a development notebook to ensure things work as intended and look right.


### 3.4 Learning Reflection

Zeroing in on the line of code where SARSA and Q-learning differ really helped me understand it better. Iterating on the sweeps comparison graphs helped me think more deeply about how to measure learning speed. And iterating on the summary tables helped me think about how to weigh three different measures to be able to draw conclusions about what best performance means.

I think week-over-week, I'm getting better at working with Claude. The engineering best practices I started with are still serving me well, but learning to prompt Claude to reason with me instead of do things for me is improving the results.

## Section 4: Speaker Notes

- Problem(s): 
    - Investigate the difference between on-policy SARSA and off-policy Q-learning
    - Understand the effects produced by different hyperparameter values for α and ε decay
    - Evaluate agent performance across multiple measures
- Method:
    - Implemented SARSA and Q-learning agents with configurable α values and injectable ε decay schedules
    - Performed benchmarked and controlled experiments for a range of α values and two decay schedules (Linear, Exponential)
- Design choice: 
    - Evaluated learning curves using a smoothed averages over 10 episodes
    - Introduced a performance index achieved by normalizing and weighting learning speed, final performance, and sample efficiency
- Key result: 
    - Agents with ε Exponential Decay produce the best results
    - Sutton and Barto's published results were confirmed by every test
- Insight:
    - Agents will converge to near-optimal policy given ε-greedy parameters
    - Off-policy agents do indeed perform worse in training, but optimally in test/inference.
- Challenge: Catching conceptual mistakes requires careful inspection; no errors or warnings to help
- Connection: This lab makes TD policy control, SARSA, and Q-learning crystal clear. It set us up well to deal with the scaling constraints of tabular methods addressed in the next unit, Function Approximation.